In [1]:
import numpy as np
import rebound
import matplotlib.pyplot as plt
import os
import pandas as pd
from datetime import datetime

os.chdir('../src/')

import make_files
import integrate

import prop_elem
import sbdynt

# Initializing a Small Body Object

SBDynT is built around small_body objects, which can be used to ismualte and dynamically analyze Solar System Small Bodies.

A small_body object for which proper elements will be comptued is primarily initialized using two variables: 

(1) designation (str): This is the designation of the small body you want to analyze. This designation will be used to request orbital information form the MPC by way of JPL Horizons, so make sure that the designation you choose is a valid id which can effectively query the JP Horizons database.

(2) object_type (str): Currently there are two acceptable values for this variable, 'tno' and 'asteroid'. Using the appropriate object_type will allow SBDynT to use accurate default settings for integration length and output schema. 

Some additional variabels that can be defined by the user include:

(3) clones (int): The number of clones you want to include in your integration. Default = 0.

(4) savefile (boolean): Whether or not you want to save the initialization simulation produced for your integrations to a binary file. Default = False.

(5) filetype (str): The output folder in the ``data`` directory for your integration data. Default: 'Single', indicating that this object is analyzed by itself, rather than in some collection of objects.

We show an example of initializing small_body objects for the dwarf planets ``Ceres`` and ``Eris``. We define the ceres particle as an asteroid, and the eris particle as a tno. We also define savefile as True to save the relevant files.

In [2]:
ceres = sbdynt.small_body('Ceres',object_type = 'asteroid', savefile = True)
eris = sbdynt.small_body('Eris',object_type = 'tno', savefile = True)

The initialization of the particle does very little, and produces no outputs to begin with. 

To compute synthetic proper elements, however, we need to produce an initial REBOUND simulation of the Solar System accompanied by the particles of interest. This is done using the ``init_pe`` function within the small_body object. 

The default SBDynT run for an asteroid and TNO is produced simply by calling the function itself as shown below.

Print statements indicate which planets have been included in the simulation. The default case for an asteroid is to include every planet besides Mercury in the simulation, and the default for a tno is to include just the 4 gas giants.

In [3]:
ceres.init_pe()
eris.init_pe()

Adding:  {2: 'venus', 3: 'earth', 4: 'mars', 5: 'jupiter', 6: 'saturn', 7: 'uranus', 8: 'neptune'}
Adding:  {5: 'jupiter', 6: 'saturn', 7: 'uranus', 8: 'neptune'}


Note that since we have defined ``savefile`` as True for the ceres and eris particles, new directories have been created in "SBDynT/data/Single" representing each obejct. This directory will contain the binary files related to the analysis. 
Indeed, with the intiialization, you should see new ``archive_init.bin`` files populated within the Ceres and Eris directories. 

If ``savefile`` were left as False, the directories would be created, but the ``archive_init.bin`` files would not, as the simulation would be stored natively within the small_body object.

These files will be used to run all of the integrations in our analysis.

The user may also define two variables in the ``init_pe`` function call if they wish...

    (1) planets (None, int=(1,2,3), or dict)         default: None
            This variable can be defined in 3 ways. 
            
            None: SBDynT uses the default planet configuration for the ``object_type`` of the small_body.
            1: simulation planets = {2: 'venus', 3: 'earth', 4: 'mars', 5: 'jupiter', 6: 'saturn', 7: 'uranus', 8: 'neptune'}
            2: simulation planets = {5: 'jupiter', 6: 'saturn', 7: 'uranus', 8: 'neptune'}
            3: simulation planets = {1: 'mercury', 2: 'venus', 3: 'earth', 4: 'mars', 5: 'jupiter', 6: 'saturn', 7: 'uranus', 8: 'neptune'}
            dict: The user may define their own planet configuration by following the format of the dictionaries shown above. Example: planets = {8: 'neptune'} would initialize the simulation with the Sun, Neptune, and the small body. This type of configuration may be relevant for very detached TNOs which the user wishes to integrate for very long timescales. 
    
    (2) filename (str)                               default: 'Single'
            This variable defines the direcotry where data should be stored for the integration. Example: filename = 'Single' will save REBOUND archive binary files in data/Single/``designation``/.

# KV: Do we provide support for including Ceres or Pluto, which are relevant to some users?

# Running the Integration

Now that our simulations are initialized, we can run the integration using the ``integrate`` function.

Again, default settings allow the user to simply call the function, which will automatically set values such as the integration length and output resolution. 

However, the user may wish to set some vairbales themsevles. The integrate function has 4 variables the user can set:

    (1) tmax (float)           units: yr     default: asteroids=1e7 ; tnos = 1.5e8
        - tmax is the total length of the integration in Earth years you want to run for. 
        
    (2) tout (float)           units: yr     default: asteroids=500 ; tnos = 5000
        - tout is the output cadence you wish to save the data in. Example: tout = 500 would save the orbital state every 500 years. For a 10 Myr integration, this would equate to 20,001 total outputs, including t=0.
        
    (3) direction (str)                      default: 'both'          options = 'forwards', 'backwards', 'both'
        - The direction of the integration. 'both' produces both a backwards and a forwards integration, integrating the particle for 1/2 of the tmax in each direction. 
        
    (4) deletefile (boolean)                  default: True
        - This variable tells the REBOUND integrator whether to delete any files of the same name. Users who may have had a failure during an integration may set this to True to start the integration at the same point that it originally failed. 

We show examples integrating both Ceres and Eris, setting tmax for shorter times, as the full integration takes 10's of minutes. 

In [4]:
ceres.integrate(tmax = 1e5)
eris.integrate(tmax = 1e6)

Integration Completed; run took  0:00:22.226394  seconds.
Integration Completed; run took  0:00:18.286501  seconds.


# Computing Synthetic Proper Elements

To compute synthetic proper elements for the small_body the typical user will simply call the ``compute_proper`` function, as shown below.
The function call will print the comptued proper elements out; additional outputs, such as the uncertianties, are saved to the particle and can be retrieved directly. 

In [5]:
ceres.compute_proper()
eris.compute_proper()

Proper Elements: {'a': 2.767073559146601, 'e': 0.36980792041022914, 'sinI': 0.16966070893125998, 'omega': 1.4253879133515488, 'Omega': 1.3531436782404813, 'g': 0.0, 's': -64.48154125361435}
Proper Elements: {'a': 67.92937526526661, 'e': 0.004192689163605993, 'sinI': 0.019515271114394443, 'omega': -2.9275952474788434, 'Omega': 0.5732165019076633, 'g': 109.61204349955649, 's': -9.026874170551709}


In [6]:
print(ceres.proper_elements)
print(ceres.proper_errors)

{'a': 2.767073559146601, 'e': 0.36980792041022914, 'sinI': 0.16966070893125998, 'omega': 1.4253879133515488, 'Omega': 1.3531436782404813, 'g': 0.0, 's': -64.48154125361435}
{'RMS_a': 2.432451275762301e-05, 'RMS_e': 0.13994735998688954, 'RMS_sinI': 0.002642454607252868}


# Chaos Indicators

Chaos indicators can be produced in much the same way as the proper elements. 3 of the chaos indicators provided by SBDynT can be computed directly from the default small_body integrations for proper elements, namely:
    (1) ACFI
    (2) Entropy
    (3) Power Distribution

The Root Mean Square indciator can also be produced from the integraiton if clones were included in the above integration. 

In addition, if proper elements have already been computed, the Distance Metric chaos indicator cna be generated directly from the values contained within th small_body object. 



Choas indicators can be computed generally using the ``compute_chaos`` function, which will compute choas indciators for every indicator which can be computed form the given simulation. For example, let us look at the process of computing choas indicators with the ``albion`` small_body object below.

In [7]:
albion = sbdynt.small_body('Albion',object_type = 'tno', savefile = True)
albion.compute_chaos()
print(albion.chaos_indicators)

{'ACFI': None, 'Entropy': None, 'Power': None, 'Distance Metric': None, 'Clone_RMS_a': None, 'Clone_RMS_e': None, 'Clone_RMS_sinI': None}


Note that the ceres.chaos_indicators dictionary currently shows empty values for each of the indicators. Running compute_chaos before producing an integration will populate none of the indicators, since nothing can be computed from the present information. 

In [8]:
albion.init_pe()
albion.integrate(tmax = 1e6)
albion.read_orbel_to_small_body()
albion.compute_chaos()
print(albion.chaos_indicators)

Adding:  {5: 'jupiter', 6: 'saturn', 7: 'uranus', 8: 'neptune'}
Integration Completed; run took  0:00:11.064048  seconds.
{'ACFI': 0.95, 'Entropy': 0.9766267343063916, 'Power': 0.8354810929711163, 'Distance Metric': None, 'Clone_RMS_a': None, 'Clone_RMS_e': None, 'Clone_RMS_sinI': None}


Now that we have run an integration, there is enough data present to comptue the ACFI, Entropy, and Power chaos indicators.

However, since proper elements have not yet been computed, the Distance Metric indicator remains empty.

The Clone RMS indicators are also empty as we did not integrate any clones. Let us do so now.

In [9]:
albion = sbdynt.small_body('Albion',object_type = 'tno', savefile = True, clones=2)
albion.init_pe()
albion.integrate(tmax = 1e6)
albion.read_orbel_to_small_body()
albion.compute_chaos()
print(albion.chaos_indicators)

Adding:  {5: 'jupiter', 6: 'saturn', 7: 'uranus', 8: 'neptune'}
Integration Completed; run took  0:00:17.081063  seconds.
{'ACFI': 0.95, 'Entropy': 0.9766267343063916, 'Power': 0.8354810929711163, 'Distance Metric': None, 'Clone_RMS_a': 0.0022913556767651857, 'Clone_RMS_e': 0.0012346911623443242, 'Clone_RMS_sinI': 4.426934103831764e-05}


In [10]:
albion.compute_proper()
albion.compute_chaos()
print(albion.chaos_indicators)

Proper Elements: {'a': 43.988791340732696, 'e': 0.20227262110040456, 'sinI': 0.008599214246128476, 'omega': -0.16012915218109774, 'Omega': 0.18217429744097202, 'g': 0.0, 's': -9.026874170551709}
{'ACFI': 0.95, 'Entropy': 0.9766267343063916, 'Power': 0.8354810929711163, 'Distance Metric': 245.77010211173217, 'Clone_RMS_a': 0.0022913556767651857, 'Clone_RMS_e': 0.0012346911623443242, 'Clone_RMS_sinI': 4.426934103831764e-05}
